## Ejercicio 11

Retomamos los resultados de los ejercicios anteriores para responder la reflexión final. Usamos la semilla 67, igual que en los ejercicios 6, 8 y 9. `IPython.display` se usa solamente para mostrar las tablas con un formato legible.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy

from IPython.display import Markdown, display


In [2]:
# Reutilizamos el fantoma de 8 por 8 píxeles y lo convertimos en un vector por filas.
X_fantoma = np.array([
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
    [0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
    [0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
    [0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
    [0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
    [0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)

n = X_fantoma.shape[0]
x_fantoma = X_fantoma.reshape(-1)


In [3]:
def mascara_rayo(n, direccion, s):
    # Marcamos con 1 los píxeles que cumplen la ecuación del rayo.
    alfa = direccion[0]
    beta = direccion[1]
    matriz = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            if alfa * i + beta * j == s:
                matriz[i][j] = 1
    return matriz


def matriz_proyeccion(n, direcciones):
    filas_A = []
    rayos = []

    for direccion in direcciones:
        alfa = direccion[0]
        beta = direccion[1]
        valores_s = set()

        # Reunimos los desplazamientos de esta dirección sin repetirlos.
        for i in range(n):
            for j in range(n):
                valores_s.add(alfa * i + beta * j)

        # Cada máscara vectorizada forma una fila de A.
        for s in sorted(valores_s):
            mascara = mascara_rayo(n, direccion, s)
            filas_A.append(mascara.reshape(-1))
            rayos.append((direccion, s))

    A = np.array(filas_A, dtype=float)
    return A, rayos


In [4]:
# Construimos las matrices con dos, cuatro y siete direcciones para comparar sus rangos.
D2 = [(1, 0), (0, 1)]
D4 = D2 + [(1, 1), (1, -1)]
D7 = D4 + [(2, 1), (1, 2), (3, 1)]

A2, rayos2 = matriz_proyeccion(n, D2)
A4, rayos4 = matriz_proyeccion(n, D4)
A7, rayos7 = matriz_proyeccion(n, D7)


In [5]:
def matriz_diferencias(n):
    filas_h = []
    filas_v = []

    # Las primeras filas calculan diferencias con el vecino de la derecha.
    for i in range(n):
        for j in range(n - 1):
            k = i * n + j
            fila = np.zeros(n**2)
            fila[k] = -1
            fila[k + 1] = 1
            filas_h.append(fila)

    # Las siguientes calculan diferencias con el vecino de abajo.
    for i in range(n - 1):
        for j in range(n):
            k = i * n + j
            fila = np.zeros(n**2)
            fila[k] = -1
            fila[k + n] = 1
            filas_v.append(fila)

    return np.array(filas_h + filas_v, dtype=float)


In [6]:
# Generamos las mismas mediciones ideales y el mismo ruido que en los ejercicios anteriores.
b7 = A7 @ x_fantoma
rng = np.random.default_rng(67)
z = rng.standard_normal(b7.size)
escala = np.sqrt(np.mean(b7**2))


In [7]:
# Preparamos la matriz de diferencias y las mediciones con un nivel de ruido de 0.06.
D = matriz_diferencias(n)
sigma = 0.06
b_sigma = b7 + sigma * escala * z

# Calculamos la reconstrucción por mínimos cuadrados y su residuo.
x_rec = np.linalg.lstsq(A7, b_sigma, rcond=None)[0]
r_sigma = b_sigma - A7 @ x_rec


### Inciso a)


La reconstrucción tomográfica es un problema inverso porque conocemos las mediciones y buscamos la imagen que las produjo. Se conocen la matriz de proyección $A$ y el vector de mediciones $b$, y se busca el vector $x$ con las atenuaciones de los píxeles.

El modelo $b=Ax$ relaciona ambas cosas: si conocemos una imagen, podemos calcular sus mediciones. En la reconstrucción usamos esa relación para buscar la imagen a partir de las mediciones.


### Inciso b)


In [8]:
# Calculamos el rango y la nulidad de las tres matrices, como en el ejercicio 4.
nombres = ["A2", "A4", "A7"]
matrices = [A2, A4, A7]
tabla_rangos = "| Matriz | Dimensiones | Rango | Nulidad |\n"
tabla_rangos += "| --- | --- | --- | --- |\n"

for i in range(len(matrices)):
    A = matrices[i]
    rango = np.linalg.matrix_rank(A)
    nulidad = A.shape[1] - rango

    # Agregamos una fila a la tabla con los valores calculados.
    fila = [nombres[i], A.shape, rango, nulidad]
    for valor in fila:
        tabla_rangos += "| " + str(valor) + " "
    tabla_rangos += "|\n"

display(Markdown(tabla_rangos))


| Matriz | Dimensiones | Rango | Nulidad |
| --- | --- | --- | --- |
| A2 | (16, 64) | 15 | 49 |
| A4 | (46, 64) | 39 | 25 |
| A7 | (119, 64) | 64 | 0 |


Para que las mediciones determinen una única imagen, el rango debe ser igual a la cantidad de píxeles:

$$
\operatorname{rango}(A)=n^2
\quad\text{y}\quad
\dim(\operatorname{Nul}(A))=0.
$$

Si existe un vector $v\ne0$ tal que $Av=0$, entonces $A(x+v)=Ax$. Por eso dos imágenes distintas pueden producir las mismas mediciones y no hay unicidad.

En nuestros resultados, `A2` y `A4` tienen nulidades 49 y 25. En cambio, `A7` tiene rango 64 y nulidad cero, por lo que permite recuperar de manera única el fantoma sin ruido. Tener muchas mediciones no alcanza si algunas son redundantes.


### Inciso c)


In [9]:
# Medimos el ruido agregado, el residuo y el error respecto del fantoma.
print("Nivel de ruido sigma:", sigma)
print("Norma del ruido:", np.linalg.norm(b_sigma - b7))
print("Norma del residuo:", np.linalg.norm(r_sigma))
print("Error relativo:", np.linalg.norm(x_rec - x_fantoma) / np.linalg.norm(x_fantoma))

# Este producto cercano a cero verifica que el residuo es ortogonal a las columnas de A7.
print("Norma de A7.T @ r_sigma:", np.linalg.norm(A7.T @ r_sigma))


Nivel de ruido sigma: 0.06
Norma del ruido: 0.661547500178014
Norma del residuo: 0.43834433692971386
Error relativo: 0.12896490758567475
Norma de A7.T @ r_sigma: 6.212536955172913e-14


El ruido modifica las mediciones y, en general, hace que no exista una imagen que las reproduzca exactamente. Mínimos cuadrados busca una imagen que haga lo más pequeña posible la diferencia entre las mediciones calculadas y las observadas:

$$
\min_x \|Ax-b_\sigma\|^2.
$$

El residuo $r=b_\sigma-A\hat x$ es esa diferencia. Como $A^Tr\approx0$, el residuo es ortogonal a las columnas de $A$ y $A\hat x$ es la proyección ortogonal de las mediciones sobre $\operatorname{Col}(A)$.

Con `sigma = 0.06`, la norma del residuo es aproximadamente 0,438 y el error relativo es 0,129. El residuo por sí solo no permite evaluar la calidad de la imagen: en el ejercicio 5, `A2` daba un residuo casi nulo, pero un error relativo de aproximadamente 0,469. Además, ajustar demasiado las mediciones puede implicar ajustar parte del ruido.


### Inciso d)


In [10]:
# Para resumir la regularización, resolvemos la ecuación que se obtiene al igualar el gradiente a cero.
# Es el sistema (H + lam * D.T @ D) @ x = A7.T @ b para cada caso.
H = A7.T @ A7
DtD = D.T @ D
niveles_ruido = [0.0, 0.1, 0.2]
lambdas = [0.0, 0.1, 0.3, 1.0]
resumen_regularizacion = {}

tabla_regularizacion = "| sigma | lambda | Error relativo | Residuo | Variación |\n"
tabla_regularizacion += "| --- | --- | --- | --- | --- |\n"

for sigma_tabla in niveles_ruido:
    # Usamos el mismo vector z y cambiamos solamente la intensidad del ruido.
    b_tabla = b7 + sigma_tabla * escala * z
    for lam in lambdas:
        matriz_sistema = H + lam * DtD
        termino_derecho = A7.T @ b_tabla
        x = np.linalg.solve(matriz_sistema, termino_derecho)

        # Comparamos el error, el ajuste a las mediciones y las diferencias entre vecinos.
        error = np.linalg.norm(x - x_fantoma) / np.linalg.norm(x_fantoma)
        residuo = np.linalg.norm(A7 @ x - b_tabla)
        variacion = np.linalg.norm(D @ x)
        resumen_regularizacion[sigma_tabla, lam] = {"error": error, "residuo": residuo,
                                                   "variacion": variacion}

        # Redondeamos únicamente los valores que mostramos en la tabla.
        fila = [sigma_tabla, lam, round(error, 3), round(residuo, 3), round(variacion, 3)]
        for valor in fila:
            tabla_regularizacion += "| " + str(valor) + " "
        tabla_regularizacion += "|\n"

display(Markdown(tabla_regularizacion))


| sigma | lambda | Error relativo | Residuo | Variación |
| --- | --- | --- | --- | --- |
| 0.0 | 0.0 | 0.0 | 0.0 | 2.706 |
| 0.0 | 0.1 | 0.042 | 0.201 | 2.542 |
| 0.0 | 0.3 | 0.097 | 0.504 | 2.312 |
| 0.0 | 1.0 | 0.2 | 1.159 | 1.873 |
| 0.1 | 0.0 | 0.215 | 0.731 | 3.036 |
| 0.1 | 0.1 | 0.154 | 0.778 | 2.761 |
| 0.1 | 0.3 | 0.132 | 0.956 | 2.446 |
| 0.1 | 1.0 | 0.2 | 1.495 | 1.928 |
| 0.2 | 0.0 | 0.43 | 1.461 | 3.595 |
| 0.2 | 0.1 | 0.321 | 1.506 | 3.144 |
| 0.2 | 0.3 | 0.243 | 1.661 | 2.689 |
| 0.2 | 1.0 | 0.235 | 2.126 | 2.04 |


El término $\|Dx\|^2$ supone que los píxeles vecinos suelen tener valores parecidos. Al penalizar sus diferencias, favorece imágenes más suaves.

Sin ruido, la regularización empeora la recuperación exacta del fantoma. Con `sigma = 0.1`, el mejor valor probado es `lambda = 0.3`: el error baja aproximadamente de 0,215 a 0,132. Con `sigma = 0.2`, el mejor es `lambda = 1`, que reduce el error de 0,430 a 0,235. En estas pruebas conviene regularizar más cuando aumenta el ruido.

Una regularización excesiva también suaviza los bordes y borra detalles reales. Por ejemplo, con `sigma = 0.1`, subir de `lambda = 0.3` a `lambda = 1` aumenta el error de 0,132 a 0,200. En el ejercicio 10 elegimos `lambda = 30` combinando la comparación con el ruido estimado y la observación de las estructuras de la imagen.


### Inciso e)


In [11]:
def gradiente_descendiente(A, b, alpha, max_iter=10000, tol=1e-10):
    # Comenzamos con una imagen de ceros y contamos las actualizaciones.
    x = np.zeros(A.shape[1])
    iteraciones = 0
    motivo = "máximo de iteraciones"

    for t in range(max_iter + 1):
        grad = A.T @ (A @ x - b)
        norma_grad = np.linalg.norm(grad)

        # Si el gradiente ya es pequeño, terminamos sin hacer otro paso.
        if norma_grad <= tol:
            motivo = "tolerancia del gradiente"
            break

        # Respetamos el máximo de actualizaciones permitido.
        if t == max_iter:
            break

        x = x - alpha * grad
        iteraciones += 1

    return {"x": x, "iteraciones": iteraciones, "motivo": motivo}


In [12]:
# Usamos la misma tasa y tolerancia que en el ejercicio 8.
mu_max = np.linalg.eigvalsh(H)[-1]
alpha = 1 / mu_max
resultado_gd = gradiente_descendiente(A7, b_sigma, alpha)
x_gd = resultado_gd["x"]

# Aplicamos un solo paso de Newton, partiendo también de una imagen de ceros.
x0 = np.zeros(n**2)
gradiente_inicial = A7.T @ (A7 @ x0 - b_sigma)
p0 = np.linalg.solve(H, gradiente_inicial)
x_newton = x0 - p0

# Calculamos la solución de lstsq para compararla con los métodos iterativos.
x_lstsq = np.linalg.lstsq(A7, b_sigma, rcond=None)[0]
print("Tasa de gradiente descendiente:", alpha)
print("Iteraciones de gradiente descendiente:", resultado_gd["iteraciones"])
print("Motivo de parada:", resultado_gd["motivo"])
print("Iteraciones de Newton: 1")
print()

nombres = ["Gradiente descendiente", "Newton", "lstsq"]
soluciones = [x_gd, x_newton, x_lstsq]

# Comparamos el costo y la norma del gradiente para verificar que llegan al mismo mínimo.
for i in range(len(soluciones)):
    x = soluciones[i]
    residuo = A7 @ x - b_sigma
    costo = 0.5 * np.dot(residuo, residuo)
    norma_grad = np.linalg.norm(A7.T @ residuo)

    print(nombres[i])
    print("Costo:", costo)
    print("Norma del gradiente:", norma_grad)
    print("Distancia a lstsq:", np.linalg.norm(x - x_lstsq))
    print()


Tasa de gradiente descendiente: 0.027424756403856825
Iteraciones de gradiente descendiente: 1434
Motivo de parada: tolerancia del gradiente
Iteraciones de Newton: 1

Gradiente descendiente
Costo: 0.09607287885917531
Norma del gradiente: 9.978479889076394e-11
Distancia a lstsq: 1.942730700206771e-10

Newton
Costo: 0.09607287885917523
Norma del gradiente: 1.733347233624306e-14
Distancia a lstsq: 7.707163492239222e-15

lstsq
Costo: 0.09607287885917526
Norma del gradiente: 6.212536955172913e-14
Distancia a lstsq: 0.0



`lstsq` calcula una solución de mínimos cuadrados sin que tengamos que elegir una tasa de aprendizaje. Si existen varias soluciones, devuelve la de menor norma.

Gradiente descendiente necesita una tasa, una tolerancia y un máximo de iteraciones. Se acerca al mínimo mediante pasos sucesivos. En el ejercicio 10 pudimos aplicarlo usando productos con matrices dispersas.

Newton usa el gradiente y la Hessiana para calcular el paso. En este problema la función es cuadrática y la Hessiana es constante e invertible: la aproximación que usa Newton coincide con la función completa, por eso llega al mínimo en una sola iteración.

Los costos y las distancias a `lstsq` muestran que los tres métodos obtienen prácticamente la misma solución. Ninguno elimina el ruido por sí solo: para favorecer una imagen más suave hay que incorporar la regularización al problema.


### Inciso f)


El modelo $b=Ax$ relaciona la imagen con las mediciones. El **rango** indica cuántas columnas son linealmente independientes y permite analizar la unicidad. El **espacio nulo** reúne los cambios de la imagen que no modifican las mediciones: si contiene un vector no nulo, distintas imágenes pueden producir el mismo resultado.

Con ruido, **mínimos cuadrados** busca la imagen cuyas mediciones se acercan más a las observadas. La **proyección ortogonal** explica esa aproximación: $A\hat x$ pertenece a $\operatorname{Col}(A)$ y el residuo queda perpendicular a ese espacio.

La **regularización** agrega una penalización de las diferencias entre vecinos:

$$
\min_x \frac12\left(\|Ax-b\|^2+\lambda\|Dx\|^2\right).
$$

Así equilibramos el ajuste a las mediciones con la suavidad de la imagen. En el fantoma conocemos la imagen original y podemos medir el error. En la imagen desconocida, la elección debe apoyarse en el ruido estimado, el residuo y la conservación de las estructuras.
